# Latih detektor penyakit strawberry (YOLO26) di Kaggle

Notebook ini menggantikan `train.py`. `realtime.py` **tidak** dijalankan di sini: Kaggle tidak punya kamera maupun layar, dan berjalan di server cloud sehingga tidak bisa menjangkau ESP32-CAM di jaringan Anda. Realtime tetap dijalankan di laptop (VSCode) memakai `best.pt` dari notebook ini.

**Atur dulu di panel Settings (sisi kanan), sebelum menjalankan sel apa pun:**

1. **Accelerator**: GPU T4 x2. Skrip hanya memakai satu GPU dan itu sudah cukup. Kalau muncul error CUDA "no kernel image", ganti ke T4.
2. **Internet**: On. Diperlukan untuk `pip install` dan mengunduh bobot awal `yolo26s.pt`. GPU dan internet di Kaggle mengharuskan akun terverifikasi nomor telepon.
3. **Add Input**: tambahkan dataset strawberry. Jika belum ada di Kaggle: Create → New Dataset → unggah `archive.zip` (Kaggle mengekstraknya otomatis).

Lalu jalankan sel dari atas ke bawah. Untuk training panjang, pakai **Save Version → Save & Run All** supaya tetap berjalan walau tab ditutup.

In [ ]:
# 1) Pasang Ultralytics dan cek GPU
# Jika muncul error saat import setelah pip install, restart kernel lalu lanjut dari sel ini.
!pip install -q -U ultralytics

import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU TIDAK aktif -> Settings > Accelerator > pilih GPU, lalu jalankan ulang dari sel ini.")

In [ ]:
# 2) Cari folder dataset (berisi train/, val/, test/ yang masing-masing punya images/)
import os
from pathlib import Path

def find_dataset_root(bases=("/kaggle/input", ".")):
    for base in bases:
        for dirpath, _, _ in os.walk(base, followlinks=True):
            root = Path(dirpath)
            if all((root / split / "images").is_dir() for split in ("train", "val", "test")):
                return root.resolve()
    raise FileNotFoundError("Dataset tidak ditemukan. Tambahkan dataset lewat Add Input, lalu jalankan ulang sel ini.")

DATA_DIR = find_dataset_root()
print("Dataset:", DATA_DIR)
for split in ("train", "val", "test"):
    print(f"  {split:5s}: {len(list((DATA_DIR / split / 'images').glob('*.jpg')))} gambar")

In [ ]:
# 3) Pengaturan + data.yaml (dataset tidak menyertakannya)
import yaml

# Urutan = ID kelas di file label (sudah dicocokkan dengan nama file). Jangan diubah.
CLASS_NAMES = [
    "angular_leafspot",       # 0
    "anthracnose_fruit_rot",  # 1
    "blossom_blight",         # 2
    "gray_mold",              # 3
    "leaf_spot",              # 4
    "powdery_mildew_fruit",   # 5
    "powdery_mildew_leaf",    # 6
]

MODEL   = "yolo26s.pt"  # yolo26n.pt = paling ringan, yolo26m.pt = lebih akurat
EPOCHS  = 100
IMGSZ   = 640           # gambar asli 419 px; 416 lebih cepat
BATCH   = 16
WORKERS = 4             # turunkan ke 2 jika muncul error DataLoader / shared memory
RUN     = "strawberry"

DATA_YAML = Path("data.yaml").resolve()
DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(DATA_DIR),
            "train": "train/images",
            "val": "val/images",
            "test": "test/images",
            "names": dict(enumerate(CLASS_NAMES)),
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)
print(DATA_YAML.read_text())

In [ ]:
# 4) Training. Dataset di /kaggle/input bersifat hanya-baca, jadi peringatan "cache not saved" boleh diabaikan.
import os
os.environ["WANDB_MODE"] = "disabled"  # cegah Kaggle meminta API key Weights & Biases

from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    cache=True,    # simpan gambar di RAM agar epoch lebih cepat (dataset kecil)
    patience=30,   # berhenti lebih awal kalau val tidak membaik 30 epoch
    flipud=0.5,    # daun/buah tidak punya orientasi atas-bawah yang tetap
    name=RUN,
)
BEST = Path(model.trainer.best)
print("Bobot terbaik:", BEST)

In [ ]:
# 5) Evaluasi akhir pada split test (jangan dipakai untuk tuning; lihat hanya setelah training selesai)
best = YOLO(str(BEST))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH)
print(f"\n[TEST] mAP50-95 = {metrics.box.map:.3f} | mAP50 = {metrics.box.map50:.3f}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:24s} mAP50-95 = {metrics.box.maps[i]:.3f}")

In [ ]:
# 6) Kurva training dan confusion matrix
from IPython.display import Image, display

for f in [BEST.parent.parent / "results.png", Path(metrics.save_dir) / "confusion_matrix_normalized.png"]:
    if f.exists():
        print(f.name)
        display(Image(filename=str(f)))

In [ ]:
# 7) Cek visual: prediksi pada 8 gambar test acak (judul = kelas sebenarnya dari nama file)
import random
import matplotlib.pyplot as plt

test_imgs = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))
sample = random.Random(0).sample(test_imgs, 8)
results = best.predict([str(p) for p in sample], imgsz=IMGSZ, conf=0.35, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, p, r in zip(axes.ravel(), sample, results):
    ax.imshow(r.plot()[:, :, ::-1])  # plot() menghasilkan BGR -> balik ke RGB
    ax.set_title(p.stem, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 8) Salin bobot ke /kaggle/working supaya mudah diunduh dari panel Output
import shutil
shutil.copy(BEST, "best.pt")
print("Tersimpan:", Path("best.pt").resolve(), f"({Path('best.pt').stat().st_size / 1e6:.1f} MB)")

## Setelah training

1. Unduh `best.pt` (panel **Output** di sisi kanan, atau tab Output setelah Save & Run All).
2. Taruh di folder proyek Anda, lalu jalankan `realtime.py` di laptop:

```
pip install ultralytics paho-mqtt        # paho-mqtt hanya jika memakai MQTT
python realtime.py --weights best.pt --source 0                                # webcam laptop
python realtime.py --weights best.pt --source http://IP-ESP32:81/stream --fps 1 --no-show
```

Sebelum dipasang di lapangan, ambil foto dari kamera yang benar-benar akan dipakai (ESP32-CAM) lalu latih ulang (fine-tune) dengan foto itu, karena kualitas gambarnya berbeda dari foto dataset.